# Baseline RETFound sur RFMiD / ODIR — Google Colab

Première vraie AUROC : backbone RETFound figé (poids officiels) → embeddings →
tête de classification multi-label → AUROC par classe.

**Avant de lancer** :
1. `Exécution > Modifier le type d'exécution > GPU T4`
2. Icône 🔑 (Secrets) dans la barre de gauche → ajouter `HF_TOKEN` = votre token
   HuggingFace (lecture), et activer l'accès pour ce notebook. Les conditions du
   modèle [RETFound_mae_natureCFP](https://huggingface.co/YukunZhou/RETFound_mae_natureCFP)
   doivent être acceptées sur le compte.
3. `Exécution > Tout exécuter`

RETFound est sous licence **CC BY-NC 4.0** : usage recherche uniquement.
RFMiD et ODIR sont publiques (téléchargées depuis Kaggle, sans compte).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Pas de GPU : Exécution > Modifier le type d'exécution > GPU T4"
!df -h /content | tail -1

## 1. Code et dépendances

In [ ]:
%cd /content
!git clone -q https://github.com/sumarc183-design/visual-health-platform.git || (cd visual-health-platform && git pull -q)
%cd /content/visual-health-platform
!pip install -q -r requirements.txt kagglehub

## 2. Poids RETFound (HuggingFace, dépôt gated, ~3,7 Go)

In [ ]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # lu depuis les Secrets Colab, jamais écrit dans le notebook

!python scripts/download_retfound_checkpoint.py --repo-id YukunZhou/RETFound_mae_natureCFP --out third_party/retfound/checkpoints
!ls -lh third_party/retfound/checkpoints

## 3. Données (Kaggle, téléchargement anonyme)

- `rfmid` : 1920 images train / 640 val, 28 classes retenues (~5 à 10 min de téléchargement)
- `odir` : ~6400 images, 8 classes N/D/G/C/A/H/M/O, split par patient

In [ ]:
DATASET = "rfmid"   # "rfmid" ou "odir"

import kagglehub
KAGGLE_HANDLES = {
    "rfmid": "andrewmvd/retinal-disease-classification",
    "odir": "andrewmvd/ocular-disease-recognition-odir5k",
}
DATA_ROOT = kagglehub.dataset_download(KAGGLE_HANDLES[DATASET])
print(DATA_ROOT)
!ls "{DATA_ROOT}"

## 4. Baseline : embeddings RETFound (GPU) → tête multi-label → AUROC

Images en 224×224 (résolution de préentraînement de RETFound), normalisation
ImageNet comme dans le fine-tuning officiel.

In [ ]:
import glob
CHECKPOINT = sorted(glob.glob("third_party/retfound/checkpoints/*.pth"))[0]

!python run_baseline.py \
    --dataset {DATASET} \
    --data-root "{DATA_ROOT}" \
    --checkpoint {CHECKPOINT} \
    --epochs 30 \
    --batch-size 64 \
    --num-workers 2 \
    --save-head heads/{DATASET}_head.pt 2>&1 | grep -v -i warning | tee results_{DATASET}.txt

## 5. Grad-CAM sur les vraies prédictions

Pour les classes les mieux reconnues (ou celles de `CLASSES`), prend les images
de validation réellement malades auxquelles le modèle donne la plus forte
probabilité, et montre côte à côte l'image et la zone qui a le plus pesé dans
la décision (rouge = forte contribution). À vérifier : la chaleur tombe-t-elle
sur des zones médicalement plausibles (lésions, disque optique, macula) plutôt
que sur les bords ou le fond noir ?

In [ ]:
CLASSES = []   # vide = choix automatique ; ex. ["DR", "ARMD", "MYA"] pour RFMiD, ["C", "G", "M"] pour ODIR
CLASSES_ARG = ("--classes " + " ".join(CLASSES)) if CLASSES else ""

!python scripts/gradcam_examples.py \
    --dataset {DATASET} \
    --data-root "{DATA_ROOT}" \
    --checkpoint {CHECKPOINT} \
    --head heads/{DATASET}_head.pt \
    {CLASSES_ARG} \
    --out results/gradcam_{DATASET}.png 2>&1 | grep -v -i warning

from IPython.display import Image, display
display(Image(f"results/gradcam_{DATASET}.png"))

## 6. Garder le résultat

Colab efface tout à la fin de la session : télécharger `results_rfmid.txt` /
`results_odir.txt` et `results/gradcam_<dataset>.png` (panneau Fichiers) ou les copier sur Drive. Pour l'autre
jeu, changer `DATASET` dans la cellule 3 et relancer les cellules 3 et 4.

**Lecture du chiffre** : sonde linéaire sur backbone figé, pas un fine-tuning
complet. Les AUROC publiées par RETFound sont obtenues après fine-tuning du
backbone : attendre des chiffres un peu inférieurs ici. Les classes avec très
peu de positifs en validation ont une AUROC peu fiable.

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')
# !cp results_*.txt /content/drive/MyDrive/